In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window

1. Get all completed transactions over $500 from either the US or UK (multi-condition filter).

In [0]:
df = df.filter((col('amount') > 500) & (col('status') == 'completed') & (col('region').isin('US', 'UK')))

2. Find total revenue by product and region, sorted with the highest revenue first.

In [0]:
df = df.groupBy('product_id', 'region').agg(sum('revenue').alias('total_revenue')).orderBy(col('total_revenue').desc())

3. For each customer, find their 2nd highest transaction amount (window function)

In [0]:
df = df.withColumn('rnk', dense_rank().over(Window.partitionBy('customer_id').orderBy(col('amount').desc()))).filter(col('rnk') == 2)

4. Handle missing emails and phone numbers — drop rows where both are missing, then fill the rest.

In [0]:
df = df.dropna(subset=['email', 'phone'], how='all')
df = df.fillna({'phone': 'Unknown'})

5. Join orders and customers tables, showing customer name and country with each order — including orders where the customer doesn't exist

In [0]:
df = orders.join(customers, on='cust_id', how='left').select('order_id', 'cust_id', 'name', 'country')

6. Given duplicate event IDs (some are real updates with newer timestamps), keep only the latest version of each.

In [0]:
df = df.withColumn('row_num', row_number().over(Window.partitionBy('event_id').orderBy(col('event_timestamp').desc()))).filter(col('row_num') == 1).drop('row_num')

7. Convert a string signup_date column to a real date type, then extract just the year into a new column.

In [0]:
df = df.withColumn('signup_date', col('signup_date').cast('date'))
df = df.withColumn('signup_year', year(col('signup_date')))

8. Bucket customers into age groups (Minor/Adult/Senior) — first using a UDF, then explain why when()/otherwise() is usually preferred

In [0]:
# UDF approach
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def age_bucket(age):
    if age < 18:
        return "Minor"
    elif age < 65:
        return "Adult"
    else:
        return "Senior"

age_udf = udf(age_bucket, StringType())
df = df.withColumn('age_group', age_udf(col('age')))

# Preferred approach
df = df.withColumn('age_group',
    when(col('age') < 18, "Minor")
    .when(col('age') < 65, "Adult")
    .otherwise("Senior"))

9. Explain the difference between repartition() and coalesce(), and when to use each

In [0]:
df.repartition(10)   # Full shuffle; redistributes data evenly across partitions - use when increasing partitions or fixing data skew
df.coalesce(10)       # No full shuffle; merges existing partitions — use when only reducing partition count, cheaper but can't fix skew

10. Read a JSON file into a DataFrame — explain whether you'd let Spark infer the schema or define it yourself.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
df = spark.read.schema(schema).json("path/to/file.json")